In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import ast


In [29]:
#df = pd.read_csv('/Users/oskarkarlsson/Desktop/DTU/4. Semester/Social Science/CompSci/Week3/Oskar/D2_temp_papers.csv')
df = pd.read_csv('final_papers.csv')

df['author_ids'] = df['author_ids'].apply(ast.literal_eval)#.dropna()
#Take only a half of the dataset to speed up the process
df = df.sample(frac=0.25, random_state=1)



### Create the graph

In [30]:
G = nx.Graph() # Create an empty graph

pair_counts = defaultdict(int)


for author_list in df['author_ids']: # Iterate through each paper's author list
    for i in range(len(author_list)): # Iterate through each author in the list
        for j in range(i + 1, len(author_list)): # Iterate through pairs of authors
            if author_list[i] is not None and author_list[j] is not None and author_list[i].strip() and author_list[j].strip(): # Ensure both authors are non-empty
                pair = tuple(sorted([author_list[i].strip(), author_list[j].strip()])) # Create a sorted tuple of unique author pairs
                pair_counts[pair] += 1 # Increment the count for this pair


# Convert the pair counts into a list of weighted edges for the graph
weighted_edgelist = [(a, b, count) for (a, b), count in pair_counts.items()]

# Create weighted edges in the graph
G.add_weighted_edges_from(weighted_edgelist) # Adds edges with weights in the metadata of the graph, also creates nodes
weights = [G[u][v]['weight'] for u, v in G.edges()] # Extract the weights for visualization

### Visualize the graph

In [ ]:
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.5, seed=42)  # Position nodes using the spring layout
nx.draw_networkx_nodes(G, pos, node_size=50, node_color='blue', alpha=0.7)
nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='gray', width=weights)
plt.title('Co-authorship Network')
plt.axis('off')
plt.show()

### Add attributes to nodes

In [ ]:
for node in G.nodes():
    nx.set_node_attributes(G, {node: G.degree(node)}, 'degree')
    nx.set_node_attributes(G, {node: sum(d['weight'] for _, _, d in G.edges(node, data=True))}, 'strength')

### Largest Connected Component

In [ ]:
# Extract the largest connected component
largest_cc = max(nx.connected_components(G), key=len)
LCC = G.subgraph(largest_cc).copy()

print(f"Full graph:  {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"LCC:         {LCC.number_of_nodes()} nodes, {LCC.number_of_edges()} edges")
print(f"LCC covers   {LCC.number_of_nodes() / G.number_of_nodes():.1%} of all nodes")

Full graph:  270058 nodes, 1041749 edges
LCC:         269556 nodes, 1040518 edges
LCC covers   99.8% of all nodes


### Visualize LCC with Netwulf

In [ ]:
import netwulf as nw

# Scale node size by strength for the visualization
max_strength = max(nx.get_node_attributes(LCC, 'strength').values())
for node in LCC.nodes():
    LCC.nodes[node]['size'] = LCC.nodes[node]['strength'] / max_strength * 50  # scale to reasonable range

# Identify top nodes by degree for labelling
top_n = 10
top_nodes_by_degree = sorted(LCC.nodes(), key=lambda n: LCC.nodes[n]['degree'], reverse=True)[:top_n]
for node in LCC.nodes():
    LCC.nodes[node]['label'] = node if node in top_nodes_by_degree else ''

# Launch interactive Netwulf visualization
# Adjust parameters in the UI, then close the window to get `network` and `config` back
network, config = nw.visualize(LCC, config={
    'node_size': 10,
    'node_charge': -50,
    'link_distance': 10,
    'link_width': 0.5,
    'node_size_variation': 1.0,   # uses the 'size' attribute
    'display_node_labels': False,
    'min_link_weight_percentile': 0,
    'max_link_weight_percentile': 1,
})

In [ ]:
# After closing the Netwulf window, plot the result in matplotlib for a high-res export
fig, ax = nw.draw_netwulf(network, figsize=(14, 10))
plt.title('Largest Connected Component – Co-authorship Network')
plt.savefig('lcc_netwulf.png', dpi=300, bbox_inches='tight')
plt.show()

NameError: name 'nw' is not defined